In [ ]:
import os
import re
import pandas as pd
import xml.etree.ElementTree as ET
from collections import defaultdict
from tqdm.auto import tqdm

# -----------------------------
# Paths (edit if needed)
# -----------------------------
CO_ROOT = "./WebNLG_CA"

REGISTRY_FILES = {
    "ca": "./Registry_verbalisations/registry_webnlg_en_ca.revoted.csv"
}

OVERWRITE_EMPTY = True  # if XML lex is empty but registry has text -> replace


In [44]:
# -----------------------------
# Helpers
# -----------------------------
def norm_text(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s.strip())
    return s

def strip_webnlg_prefix(p: str) -> str:
    p = norm_text(p).replace("\\", "/")
    p = re.sub(r"^\./", "", p)
    p = re.sub(r"^.*?WebNLG_ES(?:_2)?/", "", p)
    return p

def indent_xml(elem: ET.Element, level: int = 0):
    i = "\n" + level * "  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for child in elem:
            indent_xml(child, level + 1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if level and (not elem.tail or not elem.tail.strip()):
            elem.tail = i

def write_xml_atomic(tree: ET.ElementTree, path: str):
    tmp_path = path + ".tmp"
    root = tree.getroot()
    indent_xml(root, 0)
    tree.write(tmp_path, encoding="utf-8", xml_declaration=True)
    os.replace(tmp_path, path)

def find_entry_by_eid(root: ET.Element, eid: str):
    entries_parent = root.find("entries")
    if entries_parent is None:
        return None
    for entry in entries_parent.findall("entry"):
        if norm_text(entry.get("eid")) == norm_text(eid):
            return entry
    return None

def find_or_create_lex(entry: ET.Element, lang: str, lid: str) -> ET.Element:
    # STRICT match: same lang + same lid
    for lx in entry.findall("lex"):
        if norm_text(lx.get("lang")) == norm_text(lang) and norm_text(lx.get("lid")) == norm_text(lid):
            return lx

    lx = ET.Element("lex")
    lx.set("lang", lang)
    lx.set("lid", lid)
    lx.text = ""
    entry.append(lx)
    return lx

def get_first_nonempty(row, candidates):
    for c in candidates:
        if c in row and norm_text(row[c]) != "":
            return norm_text(row[c]), c
    return "", None

def load_registry(path: str, lang: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "status" in df.columns:
        df = df[df["status"].astype(str).str.upper() == "OK"].copy()

    final_col = f"final_{lang}"
    if final_col not in df.columns:
        raise ValueError(f"{path} missing column {final_col}")

    for c in df.columns:
        df[c] = df[c].map(norm_text)

    print(f"\nLoaded {path}")
    print("Columns:", list(df.columns))
    print("Rows after status filter:", len(df))

    return df

In [45]:
# -----------------------------
# Main sync
# -----------------------------
def sync_lex_from_registry(reg_path: str, lang: str):
    df = load_registry(reg_path, lang)
    final_col = f"final_{lang}"

    replaced = 0
    missing_xml = 0
    missing_entry = 0
    missing_lid = 0
    changed_files = set()

    grouped = df.groupby("xml_path", dropna=False)

    for xml_path_raw, g in tqdm(grouped, desc=f"Sync lex {lang}", unit="file"):
        rel = strip_webnlg_prefix(xml_path_raw)
        xml_path = os.path.join(CO_ROOT, rel)

        if not os.path.exists(xml_path):
            missing_xml += 1
            print(f"[missing xml] {xml_path}")
            continue

        try:
            tree = ET.parse(xml_path)
        except Exception as e:
            missing_xml += 1
            print(f"[parse error] {xml_path} -> {e}")
            continue

        root = tree.getroot()
        file_changed = False

        for _, r in g.iterrows():
            eid, eid_col = get_first_nonempty(r, ["eid", "entry_id"])
            lid, lid_col = get_first_nonempty(
                r,
                ["lid", "lex_id", "english_lid", "spanish_lid", "source_lid"]
            )
            desired = norm_text(r.get(final_col, ""))

            if not eid:
                missing_entry += 1
                continue

            if not lid:
                missing_lid += 1
                continue

            entry = find_entry_by_eid(root, eid)
            if entry is None:
                missing_entry += 1
                print(f"[missing entry] xml={xml_path} eid={eid} (from {eid_col}) lid={lid} (from {lid_col})")
                continue

            lx = find_or_create_lex(entry, lang, lid)
            before = norm_text(lx.text)

            if before != desired and (desired != "" or OVERWRITE_EMPTY):
                lx.text = desired
                replaced += 1
                file_changed = True

        if file_changed:
            write_xml_atomic(tree, xml_path)
            changed_files.add(xml_path)

    return {
        "lang": lang,
        "rows": len(df),
        "replaced": replaced,
        "missing_xml": missing_xml,
        "missing_entry": missing_entry,
        "missing_lid": missing_lid,
        "changed_files": len(changed_files),
    }

In [46]:
results = []
for lang, reg_path in REGISTRY_FILES.items():
    results.append(sync_lex_from_registry(reg_path, lang))


Loaded ./Registry_verbalisations/registry_webnlg_en_ca.revoted.csv
Columns: ['status', 'processed_at', 'xml_path', 'category', 'eid', 'source_lid', 'source_en', 'ca_nllb', 'ca_madlad', 'ca_salamandra', 'vote_threshold', 'sim_nllb_madlad', 'sim_nllb_salamandra', 'sim_madlad_salamandra', 'agree_nllb_madlad', 'agree_nllb_salamandra', 'agree_madlad_salamandra', 'voted_ca', 'final_ca', 'error']
Rows after status filter: 45116


Sync lex ca:   0%|          | 0/178 [00:00<?, ?file/s]

[missing xml] ./WebNLG_CO/dev/1triples/.ipynb_checkpoints/Airport_allSolutions-checkpoint.xml

Loaded ./Registry_verbalisations/registry_webnlg_en_eu.revoted.csv
Columns: ['status', 'processed_at', 'xml_path', 'category', 'eid', 'source_lid', 'source_en', 'eu_nllb', 'eu_madlad', 'eu_salamandra', 'vote_threshold', 'sim_nllb_madlad', 'sim_nllb_salamandra', 'sim_madlad_salamandra', 'agree_nllb_madlad', 'agree_nllb_salamandra', 'agree_madlad_salamandra', 'voted_eu', 'final_eu', 'error']
Rows after status filter: 50612


Sync lex eu:   0%|          | 0/187 [00:00<?, ?file/s]

[missing xml] ./WebNLG_CO/dev/1triples/.ipynb_checkpoints/Airport_allSolutions-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/4triples/.ipynb_checkpoints/University-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/4triples/.ipynb_checkpoints/WrittenWork-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/6triples/.ipynb_checkpoints/Astronaut-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/6triples/.ipynb_checkpoints/University-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/7triples/.ipynb_checkpoints/Astronaut-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/7triples/.ipynb_checkpoints/Monument-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/7triples/.ipynb_checkpoints/University-checkpoint.xml
[missing xml] ./WebNLG_CO/test/.ipynb_checkpoints/rdf-to-text-generation-test-data-with-refs-en-checkpoint.xml
[missing xml] ./WebNLG_CO/train/7triples/.ipynb_checkpoints/University-checkpoint.xml

Loaded ./Registry_verbalisations/registry_webnlg_en_gl.revoted.csv
Columns: ['status', 'processed_at', 'xml_path', 'categor

Sync lex gl:   0%|          | 0/187 [00:00<?, ?file/s]

[missing xml] ./WebNLG_CO/dev/1triples/.ipynb_checkpoints/Airport_allSolutions-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/4triples/.ipynb_checkpoints/University-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/4triples/.ipynb_checkpoints/WrittenWork-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/6triples/.ipynb_checkpoints/Astronaut-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/6triples/.ipynb_checkpoints/University-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/7triples/.ipynb_checkpoints/Astronaut-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/7triples/.ipynb_checkpoints/Monument-checkpoint.xml
[missing xml] ./WebNLG_CO/dev/7triples/.ipynb_checkpoints/University-checkpoint.xml
[missing xml] ./WebNLG_CO/test/.ipynb_checkpoints/rdf-to-text-generation-test-data-with-refs-en-checkpoint.xml
[missing xml] ./WebNLG_CO/train/7triples/.ipynb_checkpoints/University-checkpoint.xml


In [ ]:
print("\n===== REPLACEMENT SUMMARY =====")
for r in results:
    print(
        f"{r['lang']}: replaced={r['replaced']} | changed_files={r['changed_files']} | "
        f"missing_xml={r['missing_xml']} | missing_entry={r['missing_entry']} | rows_in_registry={r['rows']}"
    )

Check if all good

In [ ]:
# ============================================================
# Check that every XML entry has the same number of <lex> per language
# using English as reference, directly from the XML files.
#
# - Walks all XMLs under a root folder
# - For each <entry>, counts <lex lang="...">
# - Uses English ("en") count as the reference
# - Reports mismatches for the target languages
# ============================================================

from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# ----------------------------
# CONFIG
# ----------------------------
XML_ROOT = Path("WebNLG_CO")   # <- change to your XML dataset root
REF_LANG = "en"
CHECK_LANGS = ["es", "ca"]   # <- adjust to the languages present in your XMLs

# If True, only count non-empty verbalisations
COUNT_ONLY_NONEMPTY = True

# ----------------------------
# Helpers
# ----------------------------
def norm_text(x):
    return "" if x is None else " ".join(str(x).split())

def count_lex_by_lang(entry, count_only_nonempty=True):
    counts = {}
    for lx in entry.findall("lex"):
        lang = (lx.get("lang") or "").strip()
        txt = norm_text(lx.text)
        if count_only_nonempty and txt == "":
            continue
        counts[lang] = counts.get(lang, 0) + 1
    return counts

def get_entry_id(entry, fallback_idx=None):
    eid = (entry.get("eid") or "").strip()
    return eid if eid else f"entry_{fallback_idx}"

# ----------------------------
# Main scan
# ----------------------------
rows = []
xml_files = sorted(XML_ROOT.rglob("*.xml"))

print(f"Found {len(xml_files)} XML files under: {XML_ROOT}")

for xml_path in xml_files:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        rows.append({
            "xml_path": str(xml_path),
            "eid": "",
            "error": f"parse_error: {e}",
            "ref_lang": REF_LANG,
            "ref_count": None,
            "status": "parse_error",
        })
        continue

    entries_parent = root.find("entries")
    if entries_parent is None:
        rows.append({
            "xml_path": str(xml_path),
            "eid": "",
            "error": "missing <entries>",
            "ref_lang": REF_LANG,
            "ref_count": None,
            "status": "missing_entries",
        })
        continue

    for i, entry in enumerate(entries_parent.findall("entry"), start=1):
        eid = get_entry_id(entry, i)
        counts = count_lex_by_lang(entry, count_only_nonempty=COUNT_ONLY_NONEMPTY)
        ref_count = counts.get(REF_LANG, 0)

        row = {
            "xml_path": str(xml_path),
            "eid": eid,
            "error": "",
            "ref_lang": REF_LANG,
            "ref_count": ref_count,
        }

        mismatch = False
        for lang in CHECK_LANGS:
            lang_count = counts.get(lang, 0)
            row[f"count_{lang}"] = lang_count
            row[f"match_{lang}"] = (lang_count == ref_count)
            if lang_count != ref_count:
                mismatch = True

        row["status"] = "mismatch" if mismatch else "ok"
        rows.append(row)

# ----------------------------
# Results
# ----------------------------
df = pd.DataFrame(rows)

# Detailed mismatches only
detail_cols = ["xml_path", "eid", "ref_lang", "ref_count"] \
              + [f"count_{l}" for l in CHECK_LANGS] \
              + [f"match_{l}" for l in CHECK_LANGS] \
              + ["status", "error"]

df_detail = df[df["status"] != "ok"][detail_cols].copy()

# Summary per language
summary_rows = []
valid_df = df[df["status"].isin(["ok", "mismatch"])].copy()

for lang in CHECK_LANGS:
    total_entries = len(valid_df)
    mismatches = (~valid_df[f"match_{lang}"]).sum() if total_entries else 0
    missing_lang = (valid_df[f"count_{lang}"] == 0).sum() if total_entries else 0
    perfect = (valid_df[f"match_{lang}"]).sum() if total_entries else 0

    summary_rows.append({
        "lang": lang,
        "entries_checked": total_entries,
        "matches_ref_en": int(perfect),
        "mismatches_vs_en": int(mismatches),
        "missing_lang_lex": int(missing_lang),
        "match_rate": float(perfect / total_entries) if total_entries else 0.0,
    })

df_summary = pd.DataFrame(summary_rows)

# Files with at least one mismatch
if not df_detail.empty:
    df_files = (
        df_detail.groupby("xml_path", as_index=False)
        .size()
        .rename(columns={"size": "problematic_entries"})
        .sort_values(["problematic_entries", "xml_path"], ascending=[False, True])
    )
else:
    df_files = pd.DataFrame(columns=["xml_path", "problematic_entries"])

# ----------------------------
# Print
# ----------------------------
print("\n=== SUMMARY ===")
display(df_summary)

print("\n=== XML FILES WITH PROBLEMS ===")
display(df_files)

print("\n=== ENTRY-LEVEL DETAILS (mismatches / errors) ===")
display(df_detail.sort_values(["xml_path", "eid"]))
